# TP2 — Prise en main de Simu5G : analyse des résultats

Ce notebook lit les fichiers CSV produits par `tp-export` et trace les KPI.
Exécutez les cellules dans l'ordre (Shift+Entrée). Les zones `# TODO` sont à compléter.

In [ ]:
# Environnement : rend visibles les bibliothèques de l'image (pandas, matplotlib) quel que soit le noyau choisi
import sys, glob
sys.path += glob.glob('/home/opp_env/.venv/lib/python3.*/site-packages')
import sys; sys.path.append('/tp/common')
import pandas as pd, matplotlib.pyplot as plt
from tpanalyse import load_scalars, kpi_par_run
plt.rcParams['figure.figsize'] = (9, 4.5)

## 1. Un seul run : quels KPI sont disponibles ?
Après `tp-run SingleCell-DL` puis `tp-export results results.csv` :

In [ ]:
sca, iv = load_scalars('results.csv')
print(sca.name.unique())          # tous les scalaires enregistrés
sca[sca.name.str.startswith('voIP')].head(20)

**Q2.1** — Quels sont, pour chaque UE, le délai moyen (`voIPFrameDelay:mean`), la perte (`voIPFrameLoss:mean`) et le MOS (`voIPMos:mean`) ? Lequel des UE est le plus mal servi, et pourquoi (regardez sa position dans le .ini) ?

In [ ]:
kpi = sca[sca.name.isin(['voIPFrameDelay:mean','voIPFrameLoss:mean','voIPMos:mean'])]
kpi.pivot_table(index='module', columns='name', values='value')

## 2. Balayage de charge (config `Charge-DL`, runs 0..5)
Après `tp-run Charge-DL 0..5` et `tp-export results/Charge-DL charge.csv` :

In [ ]:
sca, iv = load_scalars('charge.csv')
delay = kpi_par_run(sca, iv, 'voIPFrameDelay:mean', module_filter='ue[')
loss  = kpi_par_run(sca, iv, 'voIPPlayoutLoss:mean', module_filter='ue[')   # paquets arrivés trop tard pour être joués
mos   = kpi_par_run(sca, iv, 'voIPMos:mean', module_filter='ue[')
res = delay.merge(loss, on=['run','numUEs']).merge(mos, on=['run','numUEs']).sort_values('numUEs')
res[['numUEs','voIPFrameDelay:mean','voIPPlayoutLoss:mean','voIPMos:mean']]

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].plot(res.numUEs, res['voIPFrameDelay:mean']*1000, 'o-'); ax[0].set_ylabel('Délai moyen (ms)'); ax[0].set_yscale('log')
ax[1].plot(res.numUEs, res['voIPPlayoutLoss:mean']*100, 's-r'); ax[1].set_ylabel('Perte au playout (%)')
ax[2].plot(res.numUEs, res['voIPMos:mean'], 'd-g'); ax[2].set_ylabel('MOS (1 à 4,5)')
for a in ax: a.set_xlabel('Nombre d\'UE'); a.grid(alpha=.3, which='both')
plt.suptitle('Cellule LTE 1,4 MHz (6 RB) — VoIP descendant'); plt.tight_layout(); plt.show()

**Q2.2** — À partir de quelle charge le délai décroche-t-il ? À partir de quelle charge la voix devient-elle inutilisable (MOS < 3) ? Reliez ce seuil au nombre de RB (`numBands = 6`) et au débit d'un flux VoIP (≈ 19 kbit/s reçus, cf. section 1). Combien de flux VoIP la cellule peut-elle porter ? Comparez à votre estimation de la Q1.3.

**Q2.3** — Relancez le balayage avec `numBands = 15` (3 MHz) puis `25` (5 MHz), exportez dans `charge15.csv` / `charge25.csv` et superposez les trois courbes de délai ci-dessous. La capacité suit-elle la bande ?

In [ ]:
# TODO : charger charge15.csv et charge25.csv, tracer les 3 courbes délai vs numUEs sur le même graphe (échelle log)


## 3. Mobilité (config `Mobile-DL`, runs 0..3)

In [ ]:
sca, iv = load_scalars('mobile.csv')
mob = kpi_par_run(sca, iv, 'voIPFrameDelay:mean', module_filter='ue[').sort_values('speed')
plt.plot(mob.speed*3.6, mob['voIPFrameDelay:mean']*1000, 'o-'); plt.xlabel('Vitesse (km/h)'); plt.ylabel('Délai moyen (ms)'); plt.grid(alpha=.3)

**Q2.4** — La vitesse a-t-elle un effet sur le délai ? Sur quel mécanisme de la chaîne (CQI/AMC, HARQ) joue-t-elle ? Que faudrait-il ajouter au scénario pour observer un vrai handover ?

## 4. Synthèse (à rédiger ici, 10 lignes max)
- Capacité VoIP de la cellule pour 1,4 / 3 / 5 MHz (6 / 15 / 25 RB) :
- Critère qui limite en premier (délai ou perte) :
- Ce que vous avez appris sur le lien .ini → résultats :